# Beacon WebSocket SQL Test Client

This notebook tests `/api/ws/sql` with either Basic auth handshake or ticket-based auth.

## Requirements
- `BEACON_WS_SQL_ENABLE=true`
- `BEACON_ENABLE_SQL=true`
- Python packages: `websockets`, `httpx`, `pyarrow`

In [12]:
# Optional: install dependencies in the active kernel
%pip install websockets httpx pyarrow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
import asyncio
import base64
import json
import uuid

import httpx
import pandas as pd
import pyarrow.ipc as ipc
import websockets

In [14]:
# Configuration
HOST = "127.0.0.1"
PORT = 5001
USE_TLS = False
AUTH_MODE = "basic"  # "ticket" or "basic"
USERNAME = "beacon-admin"
PASSWORD = "beacon-password"
SQL = "SELECT 1"
TIMEOUT_SECS = 30.0

In [15]:
def build_basic_auth_header(username: str, password: str) -> dict:
    token = base64.b64encode(f"{username}:{password}".encode("utf-8")).decode("ascii")
    return {"Authorization": f"Basic {token}"}


async def fetch_ws_ticket(http_base: str, username: str, password: str, timeout: float) -> str:
    headers = build_basic_auth_header(username, password)
    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.post(f"{http_base}/api/ws/sql-ticket", headers=headers)
        response.raise_for_status()
        payload = response.json()
        return payload["ticket"]


def decode_arrow_stream_to_dataframes(binary_payload: bytes):
    row_count = 0
    dataframes = []
    reader = ipc.open_stream(binary_payload)
    for batch in reader:
        row_count += batch.num_rows
        dataframes.append(batch.to_pandas())
    return row_count, dataframes


async def resolve_ws_endpoint(
    auth_mode: str | None = None,
    timeout: float | None = None,
    use_tls: bool | None = None,
    host: str | None = None,
    port: int | None = None,
    username: str | None = None,
    password: str | None = None,
):
    auth_mode = auth_mode or AUTH_MODE
    timeout = timeout or TIMEOUT_SECS
    use_tls = USE_TLS if use_tls is None else use_tls
    host = host or HOST
    port = port or PORT
    username = username or USERNAME
    password = password or PASSWORD

    scheme_http = "https" if use_tls else "http"
    scheme_ws = "wss" if use_tls else "ws"

    http_base = f"{scheme_http}://{host}:{port}"
    ws_base = f"{scheme_ws}://{host}:{port}/api/ws/sql"

    if auth_mode == "basic":
        headers = build_basic_auth_header(username, password)
        ws_url = ws_base
    elif auth_mode == "ticket":
        ticket = await fetch_ws_ticket(http_base, username, password, timeout)
        headers = None
        ws_url = f"{ws_base}?ticket={ticket}"
    else:
        raise ValueError("AUTH_MODE must be 'basic' or 'ticket'")

    return ws_url, headers, timeout


async def run_sql_dataframe(
    sql: str,
    *,
    auth_mode: str | None = None,
    timeout: float | None = None,
    verbose: bool = True,
):
    ws_url, headers, timeout = await resolve_ws_endpoint(auth_mode=auth_mode, timeout=timeout)
    request_id = str(uuid.uuid4())
    total_rows = 0
    dataframes = []

    if verbose:
        print(f"Connecting: {ws_url}")

    async with websockets.connect(ws_url, additional_headers=headers, open_timeout=timeout) as ws:
        await ws.send(json.dumps({
            "type": "run_sql",
            "sql": sql,
            "request_id": request_id,
        }))

        if verbose:
            print(f"Sent run_sql request_id={request_id}")

        while True:
            frame = await ws.recv()

            if isinstance(frame, str):
                event = json.loads(frame)
                event_type = event.get("type")

                if verbose:
                    print("text event:", event)

                if event_type == "error":
                    raise RuntimeError(event.get("message", "unknown websocket error"))
                if event_type == "done":
                    break
            else:
                batch_rows, batch_frames = decode_arrow_stream_to_dataframes(bytes(frame))
                total_rows += batch_rows
                dataframes.extend(batch_frames)

    if verbose:
        print(f"Decoded rows from binary stream: {total_rows}")

    if not dataframes:
        return pd.DataFrame()

    return pd.concat(dataframes, ignore_index=True)


async def run_sql(sql: str, **kwargs):
    """Convenience alias for notebook cells: returns a pandas DataFrame."""
    return await run_sql_dataframe(sql, **kwargs)

In [16]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql(SQL)
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=bf214860-969e-4422-b2c0-6d36f7b7cccf
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': 'bf214860-969e-4422-b2c0-6d36f7b7cccf'}
text event: {'type': 'chunk', 'request_id': 'bf214860-969e-4422-b2c0-6d36f7b7cccf', 'batch_index': 0, 'row_count': 1}
text event: {'type': 'done', 'request_id': 'bf214860-969e-4422-b2c0-6d36f7b7cccf', 'total_rows': 1}
Decoded rows from binary stream: 1


,Int64(1)
0,1


In [56]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("CREATE ATLAS TABLE example LOCATION 'atlas-example'")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=b91be17b-2ac7-4da1-8d27-5dce166f2e54
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': 'b91be17b-2ac7-4da1-8d27-5dce166f2e54'}
text event: {'type': 'done', 'request_id': 'b91be17b-2ac7-4da1-8d27-5dce166f2e54', 'total_rows': 0}
Decoded rows from binary stream: 0


""


In [57]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("DESCRIBE example")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=7367ed1f-d60c-4c62-bb0a-e32c7a49f273
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '7367ed1f-d60c-4c62-bb0a-e32c7a49f273'}
text event: {'type': 'chunk', 'request_id': '7367ed1f-d60c-4c62-bb0a-e32c7a49f273', 'batch_index': 0, 'row_count': 0}
text event: {'type': 'done', 'request_id': '7367ed1f-d60c-4c62-bb0a-e32c7a49f273', 'total_rows': 0}
Decoded rows from binary stream: 0


,column_name,data_type,is_nullable


In [58]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("DROP TABLE example ")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=85866897-cb5e-4421-be76-c65d287a1492
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '85866897-cb5e-4421-be76-c65d287a1492'}
text event: {'type': 'done', 'request_id': '85866897-cb5e-4421-be76-c65d287a1492', 'total_rows': 0}
Decoded rows from binary stream: 0


""
